# Klasifikasi Penyakit Daun Paprika menggunakan Convolutional Neural Network (CNN)

Notebook ini digunakan untuk melatih dan mengevaluasi model deep learning CNN dalam mendeteksi dan mengklasifikasikan penyakit pada daun paprika. Eksperimen dibagi menjadi tiga tahap:
1. **Baseline Model** (Model dasar tanpa augmentasi)
2. **Model dengan Online Augmentation** (Augmentasi dinamis menggunakan layer Keras)
3. **Model dengan Offline Augmentation** (Augmentasi fisik menggunakan ImageDataGenerator)

## 1. Setup & Eksplorasi Data

Pada bagian ini, kita mengimpor seluruh library yang dibutuhkan serta memeriksa direktori dataset lokal.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report

# Konfigurasi Path Dataset
dataset_path = "Pepper_Dataset"
print("TensorFlow Version:", tf.__version__)
print("Dataset Path:", os.path.abspath(dataset_path))

# Tampilkan kelas dan jumlah gambar di masing-masing folder
if os.path.exists(dataset_path):
    for folder in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder)
        if os.path.isdir(folder_path):
            print(f"- {folder}: {len(os.listdir(folder_path))} gambar")
else:
    print("Error: Folder dataset tidak ditemukan!")

### Fungsi Helper untuk Prediksi Gambar

Kita mendefinisikan sebuah fungsi helper agar proses prediksi gambar pengujian lebih ringkas dan tidak berulang.

In [ ]:
def predict_and_plot(model_obj, img_path, class_names):
    if not os.path.exists(img_path):
        print(f"Error: Gambar {img_path} tidak ditemukan!")
        return
    
    # Load & preprocess
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    prediction = model_obj.predict(img_array)
    pred_idx = np.argmax(prediction)
    predicted_class = class_names[pred_idx]
    confidence = prediction[0][pred_idx] * 100
    
    # Plot
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Prediksi: {predicted_class} ({confidence:.2f}%)")
    plt.axis("off")
    plt.show()
    
    print(f"File: {img_path}")
    print(f"Hasil Prediksi: {predicted_class} ({confidence:.2f}%)")
    print(f"Probabilitas: {list(zip(class_names, prediction[0]))}\n")

## 2. Eksperimen 1: Model (Tanpa Augmentasi)

Melatih model CNN dasar tanpa menggunakan teknik augmentasi data untuk melihat performa awal model.

In [ ]:
# Load dataset training & validation
train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Dapatkan nama kelas
class_names_inferred = train_data.class_names
print("Inferred Class Names:", class_names_inferred)

# Normalisasi pixel ke range [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
val_data = val_data.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Struktur Model
model_baseline = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_baseline.summary()

In [ ]:
model_baseline.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline = model_baseline.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

# Simpan model baseline
model_baseline.save("model_paprika.h5")

In [ ]:
# Plot accuracy & loss side-by-side
acc = history_baseline.history['accuracy']
val_acc = history_baseline.history['val_accuracy']
loss = history_baseline.history['loss']
val_loss = history_baseline.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

# Evaluasi menggunakan Confusion Matrix & Classification Report
y_true = []
y_pred = []
for images, labels in val_data:
    preds = model_baseline.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Standard class names dengan underscore untuk reporting
report_classes = ["Bacterial_spot", "Cercospora_leaf_spot", "Healthy"]
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=report_classes))

In [ ]:
# Uji model baseline pada paprika.jpg
predict_and_plot(model_baseline, "samples/paprika.jpg", report_classes)

## 3. Eksperimen 2: Model dengan Online Data Augmentation

Menambahkan layer augmentasi data bawaan Keras langsung di dalam struktur model untuk melakukan augmentasi gambar secara dinamis saat training.

In [ ]:
# Definisikan layer augmentasi dinamis
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])

model_online_aug = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_online_aug.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_online_aug = model_online_aug.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

# Simpan model
model_online_aug.save("model_paprika_augmented.h5")

In [ ]:
# Plot history
acc = history_online_aug.history['accuracy']
val_acc = history_online_aug.history['val_accuracy']
loss = history_online_aug.history['loss']
val_loss = history_online_aug.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='train acc')
plt.plot(epochs_range, val_acc, label='val acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='train loss')
plt.plot(epochs_range, val_loss, label='val loss')
plt.legend()
plt.title('Loss')
plt.show()

# Evaluasi Metrics
y_true = []
y_pred = []
for images, labels in val_data:
    preds = model_online_aug.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=report_classes))

## 4. Eksperimen 3: Offline Data Augmentation (ImageDataGenerator)

Melakukan augmentasi gambar secara offline menggunakan `ImageDataGenerator` untuk secara fisik menghasilkan dan menyimpan salinan gambar baru yang teraugmentasi langsung di dalam folder kelas `Cercospora Leaf Spot`.

In [ ]:
# Tentukan folder sumber augmentasi
source_folder = "Pepper_Dataset/Cercospora Leaf Spot"
print("Jumlah gambar Cercospora sebelum augmentasi:", len(os.listdir(source_folder)))

# Definisikan generator
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

current_images = [f for f in os.listdir(source_folder) if not f.startswith('aug')]
aug_images = [f for f in os.listdir(source_folder) if f.startswith('aug')]

if len(aug_images) == 0:
    print("Menjalankan offline data augmentation...")
    count = 0
    for filename in current_images:
        img_path = os.path.join(source_folder, filename)
        try:
            img = load_img(img_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)
            
            i = 0
            for batch in datagen.flow(
                x,
                batch_size=1,
                save_to_dir=source_folder,
                save_prefix='aug',
                save_format='jpg'
            ):
                i += 1
                count += 1
                if i >= 5:
                    break
        except Exception as e:
            print(f"Error pada {filename}: {e}")
    print("Total gambar hasil augmentasi baru:", count)
else:
    print("Augmentasi offline sudah pernah dijalankan sebelumnya (ditemukan file berawalan 'aug'). Proses dilewati.")

print("Jumlah gambar Cercospora sekarang:", len(os.listdir(source_folder)))

In [ ]:
# Reload dataset setelah penambahan gambar fisik teraugmentasi
train_data_final = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data_final = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Normalisasi
train_data_final = train_data_final.map(lambda x, y: (normalization_layer(x), y))
val_data_final = val_data_final.map(lambda x, y: (normalization_layer(x), y))

# Struktur Model Final
model_final = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:
model_final.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_final = model_final.fit(
    train_data_final,
    validation_data=val_data_final,
    epochs=20
)

# Simpan model final
model_final.save("model_paprika_final.h5")

## Evaluasi Model Final

Bagian ini menampilkan evaluasi komprehensif terhadap performa Model Final (Eksperimen 3) setelah proses training selesai.

### 1. Grafik Training & Validation (Accuracy & Loss)

In [ ]:
# Plot history
acc = history_final.history['accuracy']
val_acc = history_final.history['val_accuracy']
loss = history_final.history['loss']
val_loss = history_final.history['val_loss']
epochs_range = range(len(acc))

# Grafik Training/Validation Accuracy & Loss
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#1f77b4', linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#ff7f0e', linewidth=2)
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#1f77b4', linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#ff7f0e', linewidth=2)
plt.legend(loc='upper right')
plt.title('Training and Validation Loss', fontsize=12, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Menghitung Prediksi Dataset Validasi

Langkah awal evaluasi adalah memprediksi seluruh data validasi untuk digunakan dalam metrik berikutnya.

In [ ]:
# Evaluasi Metrics - Prediksi data validasi
y_true = []
y_pred = []
for images, labels in val_data_final:
    preds = model_final.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

### 2. Confusion Matrix (Heatmap)

Menampilkan perbandingan antara kelas asli (*actual*) dan hasil prediksi model (*predicted*).

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)

def plot_confusion_matrix_custom(cm, classes, title='Confusion Matrix Heatmap', cmap=plt.cm.Blues):
    plt.figure(figsize=(7, 6))
    im = plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.colorbar(im)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Labeling each cell
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontweight="bold", fontsize=12)

    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()

try:
    import seaborn as sns
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=report_classes, yticklabels=report_classes,
                annot_kws={"size": 12, "weight": "bold"})
    plt.title('Confusion Matrix Heatmap', fontsize=14, fontweight='bold')
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.show()
except ImportError:
    plot_confusion_matrix_custom(cm, report_classes)

### 3. Classification Report (Tabel)

Tabel ini memuat rincian performa per kelas (Precision, Recall, F1-Score, dan Support).

In [ ]:
from sklearn.metrics import classification_report
report_dict = classification_report(y_true, y_pred, target_names=report_classes, output_dict=True)

try:
    import pandas as pd
    from IPython.display import display
    df_report = pd.DataFrame(report_dict).transpose()
    print("\nClassification Report Table:")
    display(df_report)
except ImportError:
    # Print a manually formatted nice table
    print("\n" + "="*70)
    print("CLASSIFICATION REPORT TABLE".center(70))
    print("="*70)
    print(f"{'Class':<25} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Support':<8}")
    print("-"*70)
    for key, val in report_dict.items():
        if key == 'accuracy':
            print(f"{'accuracy':<25} | {'':<10} | {'':<10} | {val:<10.4f} | {int(report_dict['macro avg']['support']):<8}")
            print("-"*70)
        else:
            print(f"{key:<25} | {val['precision']:<10.4f} | {val['recall']:<10.4f} | {val['f1-score']:<10.4f} | {int(val['support']):<8}")
    print("="*70)

### 4. Bar Charts Metrik per Kelas

Visualisasi perbandingan metrik Precision, Recall, F1-Score, dan Support dalam bentuk diagram batang.

In [ ]:
classes = report_classes
metrics = ['precision', 'recall', 'f1-score', 'support']
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']

plt.figure(figsize=(16, 10))

for idx, metric in enumerate(metrics):
    plt.subplot(2, 2, idx + 1)
    values = [report_dict[c][metric] for c in classes]
    bars = plt.bar(classes, values, color=colors[idx], edgecolor='black', alpha=0.8)
    plt.title(f"{metric.capitalize()} per Class", fontsize=12, fontweight='bold')
    plt.ylabel(metric.capitalize())
    
    # Adjust y-limit for metrics (0 to 1.1) or support (automatic)
    if metric != 'support':
        plt.ylim(0, 1.15)
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
    else:
        max_val = max(values) if values else 10
        plt.ylim(0, max_val * 1.15)
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + (max_val * 0.02), f"{int(yval)}", ha='center', va='bottom', fontweight='bold')
            
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### 5. Visualisasi Beberapa Hasil Prediksi Gambar

Menampilkan beberapa sampel gambar acak dari dataset validasi beserta label asli, label hasil prediksi, dan confidence score.

In [ ]:
plt.figure(figsize=(14, 10))
# Ambil satu batch dari validation data
images_batch, labels_batch = next(iter(val_data_final.take(1)))
preds_batch = model_final.predict(images_batch)

# Tampilkan 6 gambar pertama dari batch
num_images_to_show = min(6, len(images_batch))
for i in range(num_images_to_show):
    plt.subplot(2, 3, i + 1)
    
    # Image might be normalized to [0, 1]
    img = images_batch[i].numpy()
    plt.imshow(img)
    
    true_idx = labels_batch[i].numpy()
    pred_probs = preds_batch[i]
    pred_idx = np.argmax(pred_probs)
    confidence = pred_probs[pred_idx] * 100
    
    true_label = report_classes[true_idx]
    pred_label = report_classes[pred_idx]
    
    color = 'green' if true_idx == pred_idx else 'red'
    
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.2f}%)", color=color, fontsize=10, fontweight='bold')
    plt.axis('off')

plt.suptitle('Prediction Visualization on Validation Batch', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### 6. Ringkasan Metrik Keseluruhan Model

Menghitung metrik performa global model final menggunakan weighted average.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

overall_acc = accuracy_score(y_true, y_pred)
overall_prec = precision_score(y_true, y_pred, average='weighted')
overall_rec = recall_score(y_true, y_pred, average='weighted')
overall_f1 = f1_score(y_true, y_pred, average='weighted')

print("\n" + "="*60)
print("EVALUASI KESELURUHAN MODEL (Weighted Average)".center(60))
print("="*60)
print(f"Accuracy  : {overall_acc:.4f} ({overall_acc*100:.2f}%)")
print(f"Precision : {overall_prec:.4f} ({overall_prec*100:.2f}%)")
print(f"Recall    : {overall_rec:.4f} ({overall_rec*100:.2f}%)")
print(f"F1-Score  : {overall_f1:.4f} ({overall_f1*100:.2f}%)")
print("="*60)